In [ ]:
import cv2
import os
import torch
import random
from facenet_pytorch import MTCNN
from tqdm import tqdm

# 1. Setup Device and Detector
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
mtcnn = MTCNN(image_size=224, margin=20, device=device, post_process=True)

# 2. Define Paths
BASE_DIR = "../raw_videos" 
OUT_DIR = "../processed"
CATEGORIES = ['Celeb-real', 'Celeb-fake']
LIMIT = 500  # We will only take 500 from each

def extract_faces_balanced(category, num_frames=20):
    input_path = os.path.join(BASE_DIR, category)
    
    # Get all video files and shuffle them to get a diverse sample
    all_videos = [f for f in os.listdir(input_path) if f.endswith(('.mp4', '.avi'))]
    random.seed(42) # For reproducibility
    random.shuffle(all_videos)
    
    # Select only the first 500
    selected_videos = all_videos[:LIMIT]
    
    print(f"Processing {len(selected_videos)} videos in {category}...")
    
    for video_name in tqdm(selected_videos):
        video_path = os.path.join(input_path, video_name)
        video_id = os.path.splitext(video_name)[0]
        
        save_folder = os.path.join(OUT_DIR, category, video_id)
        if not os.path.exists(save_folder): os.makedirs(save_folder)
        
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        if total_frames <= 0:
            continue
            
        interval = max(1, total_frames // num_frames)
        
        count = 0
        saved = 0
        while cap.isOpened() and saved < num_frames:
            ret, frame = cap.read()
            if not ret: break
            
            if count % interval == 0:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                save_path = os.path.join(save_folder, f"frame_{saved}.jpg")
                
                try:
                    # Detect and save crop
                    mtcnn(frame_rgb, save_path=save_path)
                    saved += 1
                except Exception:
                    pass 
            count += 1
        cap.release()

# 3. Execute for both categories
for cat in CATEGORIES:
    extract_faces_balanced(cat)

In [ ]:
import os
import cv2
import torch
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
from efficientnet_pytorch import EfficientNet

# Set seed for reproducibility
random.seed(42)
torch.manual_seed(42)

# Hardware check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Project running on: {device}")

In [ ]:

class SRMLayer(nn.Module):
    def __init__(self):
        super(SRMLayer, self).__init__()
        # 3 standard SRM filters to capture noise residuals
        srm_kernel = np.array([
            [[-1,  2, -1], [ 2, -4,  2], [-1,  2, -1]], 
            [[-1, -1, -1], [-1,  8, -1], [-1, -1, -1]], 
            [[ 0, -1,  0], [-1,  4, -1], [ 0, -1,  0]]  
        ], dtype=np.float32) / 4.0
        
        self.kernel = torch.tensor(srm_kernel).view(3, 1, 3, 3).repeat(1, 3, 1, 1)
        self.kernel = nn.Parameter(self.kernel, requires_grad=False)

    def forward(self, x):
        return F.conv2d(x, self.kernel, stride=1, padding=1)

class TwoStreamNet(nn.Module):
    def __init__(self):
        super(TwoStreamNet, self).__init__()
        # Stream 1: Spatial (Standard RGB)
        self.spatial_net = EfficientNet.from_pretrained('efficientnet-b0')
        
        # Stream 2: Noise (SRM)
        self.srm_layer = SRMLayer()
        self.noise_net = EfficientNet.from_pretrained('efficientnet-b0')
        
        # Fusion Classifier
        self.fc = nn.Sequential(
            nn.Linear(1280 * 2, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # RGB Stream Feature Extraction
        feat1 = self.spatial_net.extract_features(x)
        feat1 = F.adaptive_avg_pool2d(feat1, 1).view(feat1.size(0), -1)
        
        # Noise Stream Feature Extraction (On-the-fly SRM)
        noise = self.srm_layer(x)
        feat2 = self.noise_net.extract_features(noise)
        feat2 = F.adaptive_avg_pool2d(feat2, 1).view(feat2.size(0), -1)
        
        # Concatenation and Final Prediction
        combined = torch.cat((feat1, feat2), dim=1)
        return self.fc(combined)

# Initialize the model
model = TwoStreamNet().to(device)

In [ ]:
class CelebDFDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        
        # Iterate through Celeb-real and Celeb-fake
        for label, category in enumerate(['Celeb-real', 'Celeb-fake']):
            cat_path = os.path.join(root_dir, category)
            if not os.path.exists(cat_path): continue
            
            for vid_folder in os.listdir(cat_path):
                folder_path = os.path.join(cat_path, vid_folder)
                frames = [os.path.join(folder_path, f) for f in os.listdir(folder_path)]
                for frame_path in frames:
                    self.samples.append((frame_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(label, dtype=torch.float32)

# Preprocessing transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Initialize Dataset and Loaders
# Update path to match your D drive setup
DATA_PATH = r'D:\CHODER\Two_stream_Detection\backend\processed'
full_dataset = CelebDFDataset(root_dir=DATA_PATH, transform=transform)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False)

print(f"Dataset Loaded: {len(train_ds)} training images, {len(val_ds)} validation images.")